<a href="https://colab.research.google.com/github/bendonick/ml_nlp/blob/main/exercise1/ex1_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML4NLP1
## Starting Point for Exercise 1, part II

This notebook is supposed to serve as a starting point and/or inspiration when starting exercise 1, part II.

One of the goals of this exercise is to make you acquainted with **skorch**. You will probably need to consult the [documentation](https://skorch.readthedocs.io/en/stable/).

## Use of generative AI

AI tools may be used in this exercise to help you explain code, debug, improve documentation, and explore alternatives. Any use must be documented in the *Declaration on the use of AI* at the end of this notebook (which tools were used and for which purposes). You remain responsible for understanding, checking, and testing everything you submit, and you must be able to explain any submitted text, code, analyses, and AI-assisted contributions. See the [CL guidelines on academic integrity and AI use](https://www.cl.uzh.ch/en/Studies/Services/academic-integrity.html).

# Setup

In [1]:
# Pin the library versions used in this course (the same ones as in Part I and the W04 tutorial).
# PyTorch is preinstalled on Colab; for a local installation see https://pytorch.org/get-started/locally/
%pip install -q scikit-learn==1.9.1 skorch==1.4.0 gdown

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
from skorch import NeuralNetClassifier

import pandas as pd
import numpy as np
import csv
import re
import string
from collections import defaultdict

# Set seed for reproducibility
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

## Training a classifier and making predictions

In [3]:
# Download dataset
!gdown 1QP6YuwdKFNUPpvhOaAcvv2Pcp4JMbIRs # x_train
!gdown 1QVo7PZAdiZKzifK8kwhEr_umosiDCUx6 # x_test
!gdown 1QbBeKcmG2ZyAEFB3AKGTgSWQ1YEMn2jl # y_train
!gdown 1QaZj6bI7_78ymnN8IpSk4gVvg-C9fA6X # y_test

Downloading...
From: https://drive.google.com/uc?id=1QP6YuwdKFNUPpvhOaAcvv2Pcp4JMbIRs
To: /content/x_train.txt
100% 64.1M/64.1M [00:00<00:00, 130MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QVo7PZAdiZKzifK8kwhEr_umosiDCUx6
To: /content/x_test.txt
100% 65.2M/65.2M [00:00<00:00, 186MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QbBeKcmG2ZyAEFB3AKGTgSWQ1YEMn2jl
To: /content/y_train.txt
100% 480k/480k [00:00<00:00, 43.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QaZj6bI7_78ymnN8IpSk4gVvg-C9fA6X
To: /content/y_test.txt
100% 480k/480k [00:00<00:00, 74.9MB/s]


In [4]:
with open('x_train.txt') as f:
    x_train = f.read().splitlines()
with open('y_train.txt') as f:
    y_train = f.read().splitlines()
with open('x_test.txt') as f:
    x_test = f.read().splitlines()
with open('y_test.txt') as f:
    y_test = f.read().splitlines()

In [5]:
# Combine x_train and y_train into one dataframe
train_df = pd.DataFrame({'text': x_train, 'label': y_train})
# Combine x_test and y_test into one dataframe
test_df = pd.DataFrame({'text': x_test, 'label': y_test})
# Inspect the first 5 items in the train split
train_df.head()

,text,label
0,Klement Gottwaldi surnukeha palsameeriti ning ...,est
1,"Sebes, Joseph; Pereira Thomas (1961) (på eng)....",swe
2,भारतीय स्वातन्त्र्य आन्दोलन राष्ट्रीय एवम क्षे...,mai
3,"Après lo cort periòde d'establiment a Basilèa,...",oci
4,ถนนเจริญกรุง (อักษรโรมัน: Thanon Charoen Krung...,tha


### Data preparation

Prepare your dataset for this experiment using the same method as you did in part 1.

Get a subset of the train/test data that includes 20 languages. Include English (`eng`), German (`deu`), Dutch (`nld`), Danish (`dan`), Swedish (`swe`), Norwegian (either Bokmål `nob` or Nynorsk `nno`), and Japanese (`jpn`), plus 13 additional languages of your choice based on the items in the list of labels.

Unlike `sklearn`, the neural network cannot work with string labels: `skorch` and `CrossEntropyLoss` expect the labels as integer class indices `0, ..., n_classes - 1`. We therefore encode them with a [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html).


In [6]:
from IPython.testing import test
# TODO: Create your train/test subsets of languages
# Note, make sure these are the same as what you used in Part 1!

from sklearn.model_selection import train_test_split

all_df = pd.concat([train_df, test_df]).drop_duplicates('text')
train_df, test_df = train_test_split(all_df, test_size = 0.2, stratify=all_df['label'], random_state=42)

langs = ['eng', 'deu', 'nld', 'dan', 'swe', 'nob', 'jpn',
         'fra', 'spa', 'ita', 'por', 'fin', 'pol', 'rus',
         'ces', 'hun', 'tur', 'ell', 'ara', 'zho']

train_sub = train_df[train_df['label'].isin(langs)].reset_index(drop=True)
test_sub = test_df[test_df['label'].isin(langs)].reset_index(drop=True)

x_train, y_train = train_sub['text'].tolist(), train_sub['label'].values
x_test, y_test = test_sub['text'].tolist(), test_sub['label'].values

print(len(x_train), len(x_test))

15983 3996


In [7]:
# TODO: With the following code, we wanted to ENCODE the labels, however, our cat was walking on the keyboard and some of it got changed. Can you fix it?
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder().fit(y_train)
y_train, y_test = label_encoder.transform(y_train), label_encoder.transform(y_test)
print(label_encoder.classes_)
print(y_train)
print(y_test)

['ara' 'ces' 'dan' 'deu' 'ell' 'eng' 'fin' 'fra' 'hun' 'ita' 'jpn' 'nld'
 'nob' 'pol' 'por' 'rus' 'spa' 'swe' 'tur' 'zho']
[13 12 15 ...  1  5 13]
[14  6 18 ... 11 17  0]


### Feature Extraction

In [8]:
# First, we extract some simple features as input for the neural network
# Note: x_train and y_train must refer to your 20-language subset from above, not to the full dataset
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(analyzer='char', ngram_range=(2, 2), max_features=100, binary=True)
X = vectorizer.fit_transform(x_train)

In [9]:
# We need to change the datatype to make it play nice with pytorch
X = X.astype(np.float32)
y = y_train.astype(np.int64)

In the following, we define a vanilla neural network with two hidden layers. The output layer should have as many outputs as there are classes. In addition, it should have a nonlinearity function.

In [10]:
# TODO: In the following, you can find a small (almost) working example of a neural network.
# Unfortunately, again, the cat messed up some of the code. Please fix the code such that it is executable. (Hint: the input and output sizes look a bit weird...)

class ClassifierModule(nn.Module):
    def __init__(
        self,
        num_units=200,
        nonlin=F.relu,
        input_size = 100,
        num_classes = 20,
        hidden2 = 50,
        dropout = 0.0
    ):
        super(ClassifierModule, self).__init__()
        self.num_units = num_units
        self.nonlin = nonlin

        self.dense0 = nn.Linear(input_size, num_units)
        self.nonlin = nonlin
        self.dense1 = nn.Linear(num_units, hidden2)
        self.output = nn.Linear(hidden2, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X, **kwargs):
        X = self.dropout(self.nonlin(self.dense0(X)))
        X = self.dropout(self.nonlin(self.dense1(X)))
        X = self.output(X)
        return X


In [11]:
# Initialise the neural net classifier.
net = NeuralNetClassifier(
    ClassifierModule(
        input_size=X.shape[1],
        num_units=200,
        num_classes=len(label_encoder.classes_),
        nonlin=F.relu,
    ),
    max_epochs=20,
    criterion=nn.CrossEntropyLoss(),
    lr=0.1,
    device='cuda' if torch.cuda.is_available() else 'cpu',   # train on the GPU if Colab gives you one
)

In [ ]:
# Train the classifier
net.fit(X, y)

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        2.7662       0.3363        2.4795  1.5942
      2        1.9686       0.5759        1.4442  1.4906
      3        1.2197       0.6365        1.0361  1.5706
      4        0.9302       0.7091        0.8611  1.0418
      5        0.7792       0.7394        0.7506  1.1534
      6        0.6910       0.7663        0.6834  1.0699
      7        0.6374       0.7745        0.6413  1.1406
      8        0.6020       0.7804        0.6145  1.0266
      9        0.5771       0.7801        0.5964  1.4841
     10        0.5584       0.7826        0.5840  1.6639
     11        0.5436       0.7857        0.5750  3.2727
     12        0.5313       0.7886        0.5678  2.1395
     13        0.5207       0.7904        0.5619  1.6954
     14        0.5113       0.7923        0.5570  2.1879
     15        0.5028       0.7926        0.5538  1.2117
     16        0.4951       0.7

In [ ]:
# Experiment 1: more features
vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, 3), max_features=3000, binary=True)
X = vectorizer.fit_transform(x_train).astype(np.float32).toarray()

torch.manual_seed(seed)   # same starting weights as before, so the comparison is fair

net = NeuralNetClassifier(
    ClassifierModule(
        input_size=X.shape[1],
        num_units=200,
        num_classes=len(label_encoder.classes_),
        nonlin=F.relu,
    ),
    max_epochs=20,
    criterion=nn.CrossEntropyLoss(),
    lr=0.1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
net.fit(X, y)

In [ ]:
# Experiment 2: layer sizes increased 200/50 -> 512/128, everything else like in experiment 1
torch.manual_seed(seed)

net = NeuralNetClassifier(
    ClassifierModule(
        input_size=X.shape[1],
        num_units=512,
        hidden2=128,
        num_classes=len(label_encoder.classes_),
        nonlin=F.relu,
    ),
    max_epochs=20,
    criterion=nn.CrossEntropyLoss(),
    lr=0.1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
net.fit(X, y)

In [ ]:
# Experiment 3: activation function ReLU -> tanh, everything else as in experiment 1
torch.manual_seed(seed)

net = NeuralNetClassifier(
    ClassifierModule(
        input_size=X.shape[1],
        num_units=200,
        num_classes=len(label_encoder.classes_),
        nonlin=torch.tanh,
    ),
    max_epochs=20,
    criterion=nn.CrossEntropyLoss(),
    lr=0.1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
net.fit(X, y)

In [ ]:
# Experiment 4: regularizer, dropout 0.3 after each hidden layer, everything else as in experiment 1
torch.manual_seed(seed)

net = NeuralNetClassifier(
    ClassifierModule(
        input_size=X.shape[1],
        num_units=200,
        num_classes=len(label_encoder.classes_),
        nonlin=F.relu,
        dropout=0.3,
    ),
    max_epochs=20,
    criterion=nn.CrossEntropyLoss(),
    lr=0.1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
net.fit(X, y)

In [ ]:
# Experiment 5: early stopping (patience 5 on valid_loss, best weights restored), max 50 epochs, everything else as in experiment 1
from skorch.callbacks import EarlyStopping

torch.manual_seed(seed)

net = NeuralNetClassifier(
    ClassifierModule(
        input_size=X.shape[1],
        num_units=200,
        num_classes=len(label_encoder.classes_),
        nonlin=F.relu,
    ),
    max_epochs=50,
    criterion=nn.CrossEntropyLoss(),
    lr=0.1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    callbacks=[EarlyStopping(monitor='valid_loss', patience=5, load_best=True)],
)
net.fit(X, y)

In [ ]:
# Final model: best configuration according to validation accuracy/loss (experiment 3, tanh)
from sklearn.metrics import accuracy_score, classification_report

torch.manual_seed(seed)
best_net = NeuralNetClassifier(
    ClassifierModule(
        input_size=X.shape[1],
        num_units=200,
        num_classes=len(label_encoder.classes_),
        nonlin=torch.tanh,
    ),
    max_epochs=20,
    criterion=nn.CrossEntropyLoss(),
    lr=0.1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    verbose=0,
)
best_net.fit(X, y)

X_test = vectorizer.transform(x_test).astype(np.float32).toarray()
y_pred = best_net.predict(X_test)

print(f'test accuracy: {accuracy_score(y_test, y_pred):.4f}   (logistic regression, Part 1: 0.9865)')
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_, digits=3))

Note, you can also use `GridSearchCV` with `skorch` (it follows the `sklearn` API), but be aware that training a neural network takes much more time; a handful of hand-picked configurations, compared on the validation split that `skorch` holds out for you, is enough here.

Play around with 5 different sets of hyperparameters. For example, consider some of the following:

- layer sizes
- activation functions
- regularizers
- early stopping
- vectorizer parameters (`max_features`, `ngram_range`)

Report your best hyperparameter combination.

📝❓ What is the effect of your modifications on validation performance? Discuss potential reasons.

☝ Note, during model development, if you run into the infamous CUDA out-of-memory (OOM) error, try clearing the GPU memory either with `torch.cuda.empty_cache()` or restarting the runtime.


---

## Lab report

📝❓ Write your lab report here, addressing all questions in the notebook. For convenience, the questions are collected below; please keep your answers clearly marked.

### Questions

- 📝❓ Report your best hyperparameter combination.
- 📝❓ What is the effect of your modifications on validation performance? Discuss potential reasons.

## Declarations

Please fill in both declarations below by editing this cell (double-click it). Mark the applicable option by replacing ☐ with ☒ (copy the symbol from here: ☒) and complete the text field where required. Your reviewers will check that both declarations are present and completed.

### Contribution declaration

By submitting this work, the group declares that the contributions of the group members were:

☐ approximately equal

☐ not approximately equal

If the contributions were not approximately equal, no individual group member needs to be identified.

### Declaration on the use of AI

The group declares that any use of AI-based tools in preparing this work complied with the rules specified for this exercise.

☐ No AI-based tools were used.

☐ AI-based tools were used in accordance with the rules of the exercise.

If AI-based tools were used, briefly state the tool(s) used and the purpose(s) for which they were used:



## Lab report

**Approach:** We used the same 20 languages and the same deduplicated, stratified 80/20 split as in Part 1 (15,983 training / 3,996 test texts), so the MLP and the logistic regression are directly comparable.
Labels were encoded with a `LabelEncoder` fitted once on the training labels and applied to both splits.
In the network we replaced the hard-coded input size (100) and output size (2) with parameters for the number of features and the number of classes (20), and made the activation, the size of the second hidden layer and dropout configurable. All models were trained on a Colab T4 GPU and compared on the 20% validation split that skorch holds out from the training data. The seed is reset before every run, so the runs are reproducible (experiments 1 and 5 produce identical numbers for the first 19 epochs).

Starting from the template baseline, we changed one hyperparameter per experiment. Experiment 1 changed the vectorizer; experiments 2 to 5 each changed one more hyperparameter relative to experiment 1.

| # | Hyperparameter | Change | valid_acc | best valid_loss |
|---|---|---|---|---|
| 0 | (baseline) | char bigrams, 100 features, 200/50 units, ReLU, SGD lr 0.1, 20 epochs | 79.67% | 0.5431 |
| 1 | vectorizer | char 1-3-grams, 3000 features | 98.91% | 0.0633 |
| 2 | layer sizes | 512/128 units | 98.72% | 0.0616 |
| 3 | activation | tanh instead of ReLU | **98.97%** | **0.0594** |
| 4 | regularizer | dropout 0.3 | 98.78% | 0.0613 |
| 5 | early stopping | patience 5 on valid_loss, max 50 epochs, best epoch (15) restored | 98.87% | 0.0633 |

### Questions

- 📝❓ **Report your best hyperparameter combination.**

  Experiment 3: binary character 1-3-grams (3000 features), two hidden layers with 200 and 50 units, tanh activation, no dropout, SGD with learning rate 0.1, batch size 128, 20 epochs. It reached 98.97% validation accuracy and the lowest validation loss (0.0594). On the held-out test set it scores **98.40%** accuracy (macro F1 0.984), well above the 87% target but slightly below the best logistic regression from Part 1 (98.65%).

- 📝❓ **What is the effect of your modifications on validation performance? Discuss potential reasons.**

  *Vectorizer (+19.2 points).* This was the only change with a large effect.  Train and validation loss were almost equal (0.47 vs 0.54), so the model was underfitting: the input was the bottleneck, not the network. Character 1-3-grams add single characters (scripts and special letters such as ø, ß or kana) and trigrams (short function words and word endings), the same kind of signal the linear model relied on in Part 1.

  After this change the model overfits slightly instead: training loss drops to 0.005 while validation loss bottoms out around epoch 14-15 and then rises again. The remaining experiments therefore change things that affect capacity and overfitting. Their effects on accuracy are all within 0.25 points, about 8 of ~3,200 validation texts, so they are practically tied.

  *Layer sizes (98.72%).* 512/128 units brought no improvement. The 200/50 network already fits the training data almost perfectly, so more capacity only means more room to memorise.

  *Activation (98.97%).* tanh gave the best accuracy and loss, but the gain in accuracy is within noise. However, it differs more in the training curve: it learned faster at first (95.5% after one epoch vs 90.9% with ReLU), and its validation loss kept decreasing until the last epoch instead of rising after epoch ~14. A likely reason is that tanh's bounded output (-1 to 1) limits how extreme activations become, which acts as mild regularisation.

  *Regularizer: dropout 0.3 (98.78%).* Dropout kept the training loss higher and slightly lowered the best validation loss (0.0613 vs 0.0633), but did not change accuracy. Note that skorch measures train_loss with dropout active, so part of the higher training loss is due to the measurement.

  *Early stopping (98.87%).* Training stopped after 19 epochs and restored epoch 15. The result equals experiment 1 because 20 epochs happened to be close to the optimal point already. Its main benefit is that the number of epochs does not have to be guessed.

  *Comparison with Part 1.* The MLP does not beat the logistic regression on the test set (98.40% vs 98.65%), and its weakest classes are the same: English (F1 0.943, precision 0.900), Bokmål (0.955) and Danish (0.960). This points to properties of the data rather than of the model: Danish and Bokmål are almost identical in writing, and many texts in other languages contain English citations. The linear model had several advantages: about 348,000 features including whole-word counts versus 3,000 character n-grams, and all training texts versus the 80% left after skorch's validation split. With good n-gram features, language identification is close to linearly separable, so the non-linearity of the MLP adds little. The drop from 98.97% on validation to 98.40% on test is partly because we picked the best of six nearly tied configurations, which makes the winner's validation score slightly optimistic.

AI-tools were mostly used for debugging code.